# WeedDet v7.1 — COCO training (T1 build, cleaned)

Changes vs `weeddet_trainingV7_coco.ipynb`:
- **T1 gate in Cell 1**: asserts `atss_all_neg` (no ignore band) + `cls_hard_target` are in the Drive `weeddet_v6b.py`
- **Single CONFIG cell** (Cell 3) with `RUN_TAG` — checkpoints never collide across runs
- **One eval implementation** (Cell 5) shared by training, final eval, and raw-vs-EMA — no duplicated decode loops
- **Backbone load checks `missing` keys** (partial-load guard)
- Curves guarded against empty history; broken debug cells removed; visualization draws **GT green / pred red**

Workflow per TEST_LOG.md: 20-epoch gate first. Success = det/img fluctuating well below 200, scores not pinned at 1.00, AP@50 clearly > 0.
Before running: upload repo `models/weeddet_v6b.py` -> `MyDrive/weeddet_v2_checkpoints/` (overwrite). Runtime -> GPU.

In [ ]:
# Cell 1 — Mount Drive + import weeddet_v6b (T1 gate)
from google.colab import drive
import sys, os
drive.mount('/content/drive')

SCRIPT_DIR = '/content/drive/MyDrive/weeddet_v2_checkpoints'
assert os.path.exists(f'{SCRIPT_DIR}/weeddet_v6b.py'), \
    'Upload the repo models/weeddet_v6b.py to weeddet_v2_checkpoints/ first'
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)
for _m in [m for m in list(sys.modules) if 'weeddet' in m.lower()]:
    del sys.modules[_m]
import torch, inspect
import weeddet_v6b as wd

assert hasattr(wd, 'CocoWeedDataset'), 'old weeddet_v6b.py — re-upload from the repo'
assert 'quality_iou[pos_idx].clamp' in inspect.getsource(wd.WeedDetLoss.forward), 'F3 revert missing!'
_l = wd.WeedDetLoss()
assert getattr(_l, 'cls_hard_target', False), 'v7 hard-target fix missing — re-upload weeddet_v6b.py'
assert getattr(_l, 'atss_all_neg', False), 'T1 fix missing (ignore band still active) — re-upload weeddet_v6b.py'
del _l
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('OK:', wd.__file__, '| device:', device, '| T1 active')

In [ ]:
# Cell 2 — Unzip the pre-split COCO dataset (leakage-safe; do NOT re-split)
import zipfile, json, os
from collections import Counter

ZIP_PATH  = '/content/drive/MyDrive/weeddet_v2_checkpoints/rice_detection_coco_split.zip'
DATA_ROOT = '/content/rice_coco'
CKPT_DIR  = '/content/drive/MyDrive/weeddet_v7_checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

if not os.path.exists(f'{DATA_ROOT}/train/_annotations.coco.json'):
    print('Extracting...')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(DATA_ROOT)

for s in ['train', 'valid', 'test']:
    d  = json.load(open(f'{DATA_ROOT}/{s}/_annotations.coco.json'))
    cc = Counter(a['category_id'] for a in d['annotations'])
    print(f"{s:6s} {len(d['images']):4d} imgs | boxes per cat {dict(cc)}")
# expected: train 1079 / valid 134 / test 134

In [ ]:
# Cell 3 — CONFIG (single source of truth; log every run in TEST_LOG.md)
RUN_TAG           = 'T7d'          # test id from TEST_LOG.md — baked into checkpoint names
BACKBONE_INIT     = 'riceseg'     # 'scratch' (control) | 'riceseg' | 'imagenet' (ablation only)
RICESEG_BACKBONE  = '/content/drive/MyDrive/weeddet_v6_checkpoints/riceseg_backbone.pth'

NUM_EPOCHS        = 32            # T7d: T7's fast-anneal shape, ~60% more decay-phase time
BATCH_SIZE        = 2
BASE_LR           = 0.0005        # T7d: back to T7's proven value — schedule SHAPE matters, not level
MIN_LR            = 0.00001
MOMENTUM          = 0.9
WEIGHT_DECAY      = 0.0001
GRAD_CLIP         = 10.0          # T2 verdict: keep at 10.0
WARMUP_RATIO      = 0.05
WARMUP_FACTOR     = 0.001
IMG_SIZE          = 512
CLASS_NAMES       = ('rice',)
ANCHOR_BASE_SCALE = 6             # T4: anchors 11-152px (was 5-76px; large plants unmatchable)
LSC_K             = 7
NUM_CLASSES       = len(CLASS_NAMES)
SAVE_EVERY        = 4
AP_EVAL_EVERY     = 2
USE_ATSS          = True
USE_AMP           = torch.cuda.is_available()
USE_EMA           = True          # exonerated in T0.4 — keep ON for comparability
FREEZE_BACKBONE   = True          # T7 (LP-FT phase A): freeze backbone params + BN. riceseg init only!
EMA_DECAY         = 0.999
EVAL_SCORE_THR    = 0.01
EVAL_NMS_THR      = 0.50
EVAL_MAX_DETS     = 300
SEED              = 42

RUN_NAME  = f'weeddet_v7_{BACKBONE_INIT}_{RUN_TAG}'
BEST_CKPT = f'{CKPT_DIR}/{RUN_NAME}_best.pth'
print('Run:', RUN_NAME, '| best ->', BEST_CKPT)

In [ ]:
# Cell 4 — Datasets + loaders + sanity check
import random, numpy as np
from torch.utils.data import DataLoader

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

train_ds = wd.CocoWeedDataset(DATA_ROOT, 'train', img_size=IMG_SIZE, augment=True,
                              class_names=CLASS_NAMES)
val_ds   = wd.CocoWeedDataset(DATA_ROOT, 'valid', img_size=IMG_SIZE, augment=False,
                              class_names=CLASS_NAMES)

def collate(batch):
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch)) if batch else ([], [])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=2, pin_memory=True)

img_t, tgt = train_ds[0]
assert img_t.shape == (3, IMG_SIZE, IMG_SIZE), img_t.shape
assert len(tgt['boxes']) > 0, 'first sample has no GT boxes'
print(f"train {len(train_ds)} | valid {len(val_ds)} | sample {tuple(img_t.shape)}, {len(tgt['boxes'])} boxes  OK")

In [ ]:
# Cell 4b — GT box size stats (anchor-scale sanity; paste percentiles into TEST_LOG)
import numpy as np, json as _j
_d = _j.load(open(f'{DATA_ROOT}/train/_annotations.coco.json'))
_wh = np.array([[a['bbox'][2], a['bbox'][3]] for a in _d['annotations'] if a['category_id'] == 1])
for name, v in (('width', _wh[:, 0]), ('height', _wh[:, 1]),
                ('sqrt(area)', np.sqrt(_wh[:, 0] * _wh[:, 1]))):
    q = np.percentile(v, [5, 25, 50, 75, 95])
    print(f'{name:11s} p5={q[0]:6.1f}  p25={q[1]:6.1f}  p50={q[2]:6.1f}  p75={q[3]:6.1f}  p95={q[4]:6.1f}')
print(f'anchor span with base_scale {ANCHOR_BASE_SCALE}: '
      f'{ANCHOR_BASE_SCALE*4*1.0*0.447:.0f}px (min w) to {ANCHOR_BASE_SCALE*16*1.587/0.447:.0f}px (max h)')

In [ ]:
# Cell 5 — ONE eval + visualization implementation (used by training, final eval, raw-vs-EMA)
import subprocess; subprocess.run(['pip', 'install', 'pycocotools', '-q'])
import json, contextlib, io
from PIL import Image, ImageDraw
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def make_evaluator(ds, score_thr=EVAL_SCORE_THR, nms_thr=EVAL_NMS_THR, max_dets=EVAL_MAX_DETS):
    """Returns coco_eval(net, verbose=False) -> (ap50, ap75, ap5095, det_per_img)."""
    gt_api  = COCO(ds.ann_file)
    items   = ds.items()
    cat_ids = sorted(ds.catid_to_idx)
    idx2cat = {v: k for k, v in ds.catid_to_idx.items()}
    tf = wd.T.Compose([wd.T.ToTensor(), wd.T.Normalize(mean=wd.IMAGENET_MEAN, std=wd.IMAGENET_STD)])

    @torch.no_grad()
    def coco_eval(net, verbose=False):
        net.eval()
        dets = []
        for img_id, p, W, H in items:
            img_lb, scale, pl, pt = wd.letterbox_pil(Image.open(p).convert('RGB'), IMG_SIZE)
            cls_l, regs, anchors, ishape = net._get_logits(tf(img_lb).unsqueeze(0).to(device))
            res = net._decode(cls_l, regs, anchors, ishape,
                              score_thr=score_thr, nms_thr=nms_thr,
                              max_dets=max_dets, output_thr=score_thr, use_soft_nms=False)
            boxes  = wd.unpad_boxes(res[0]['boxes'].cpu(), scale, pl, pt)
            scores = res[0]['scores'].cpu()
            labels = res[0].get('labels', torch.zeros(len(scores), dtype=torch.long)).cpu()
            boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, W)
            boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, H)
            for (x1, y1, x2, y2), s, l in zip(boxes.tolist(), scores.tolist(), labels.tolist()):
                if x2 - x1 <= 1 or y2 - y1 <= 1: continue
                dets.append({'image_id': img_id, 'category_id': idx2cat[int(l)],
                             'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': float(s)})
        if not dets:
            return 0.0, 0.0, 0.0, 0.0
        with open('/tmp/dt.json', 'w') as f: json.dump(dets, f)
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            ev = COCOeval(gt_api, gt_api.loadRes('/tmp/dt.json'), 'bbox')
            ev.params.maxDets = [10, 100, max_dets]
            ev.params.catIds  = cat_ids
            ev.evaluate(); ev.accumulate(); ev.summarize()
        if verbose: print(buf.getvalue())
        return ev.stats[1], ev.stats[2], ev.stats[0], len(dets) / len(items)
    return coco_eval

@torch.no_grad()
def plot_predictions(net, ds, num_samples=3, score_thr=EVAL_SCORE_THR, title=''):
    """GT green / predictions red, on original-resolution images."""
    import matplotlib.pyplot as plt, random as _r
    net.eval()
    tf = wd.T.Compose([wd.T.ToTensor(), wd.T.Normalize(mean=wd.IMAGENET_MEAN, std=wd.IMAGENET_STD)])
    ann = json.load(open(ds.ann_file))
    gt_by_img = {}
    for a in ann['annotations']:
        gt_by_img.setdefault(a['image_id'], []).append(a['bbox'])
    for idx in _r.sample(range(len(ds)), min(num_samples, len(ds))):
        img_id, p, W, H = ds.items()[idx]
        img = Image.open(p).convert('RGB')
        draw = ImageDraw.Draw(img)
        for (x, y, w, h) in gt_by_img.get(img_id, []):
            draw.rectangle([(x, y), (x + w, y + h)], outline=(0, 220, 0), width=2)
        img_lb, scale, pl, pt = wd.letterbox_pil(Image.open(p).convert('RGB'), IMG_SIZE)
        cls_l, regs, anchors, ishape = net._get_logits(tf(img_lb).unsqueeze(0).to(device))
        res = net._decode(cls_l, regs, anchors, ishape, score_thr=score_thr,
                          nms_thr=EVAL_NMS_THR, max_dets=EVAL_MAX_DETS,
                          output_thr=score_thr, use_soft_nms=False)
        boxes  = wd.unpad_boxes(res[0]['boxes'].cpu(), scale, pl, pt)
        scores = res[0]['scores'].cpu()
        for (x1, y1, x2, y2), s in zip(boxes.tolist(), scores.tolist()):
            x1, y1, x2, y2 = max(0, x1), max(0, y1), min(W, x2), min(H, y2)
            if x2 - x1 <= 1 or y2 - y1 <= 1: continue   # guard AFTER clamping (T2 viz crash fix)
            draw.rectangle([(x1, y1), (x2, y2)], outline=(255, 0, 0), width=2)
            draw.text((x1 + 3, y1 + 3), f'{s:.2f}', fill=(255, 0, 0))
        plt.figure(figsize=(9, 9)); plt.imshow(img)
        plt.title(f'{title} img {img_id} — GT green ({len(gt_by_img.get(img_id, []))}), pred red ({len(boxes)})')
        plt.axis('off'); plt.show()

val_eval = make_evaluator(val_ds)
print('evaluator ready')

In [ ]:
# Cell 6 — Build model + backbone init (fresh run, no resume)
import copy

model = wd.WeedDet(num_classes=NUM_CLASSES, anchor_base_scale=ANCHOR_BASE_SCALE,
                   lsc_k=LSC_K, use_atss=USE_ATSS).to(device)
assert model.criterion.atss_all_neg and model.criterion.cls_hard_target

if BACKBONE_INIT == 'riceseg':
    sd = torch.load(RICESEG_BACKBONE, map_location='cpu')
    sd = sd.get('state_dict', sd)
    bk = {k: v for k, v in sd.items() if k.startswith('backbone.')}
    assert len(bk) > 150, f'riceseg backbone looks wrong: {len(bk)} tensors'
    missing, unexpected = model.load_state_dict(bk, strict=False)
    assert not unexpected, unexpected[:5]
    miss_bk = [k for k in missing if k.startswith('backbone.')]
    assert not miss_bk, f'PARTIAL backbone load — do not train: {miss_bk[:5]}'
    print(f'riceseg backbone: {len(bk)} tensors | ALL BN trainable (never apply_bn_policy here)')
elif BACKBONE_INIT == 'imagenet':
    wd.load_imagenet_backbone(model)               # known-bad F1+F2 combo — ablation only
    wd.apply_bn_policy(model, pretrained_loaded=True, verbose=True)
else:
    assert BACKBONE_INIT == 'scratch', BACKBONE_INIT
    print('from scratch — all BN trainable (control)')

def freeze_backbone_(m):
    """T7/LP-FT phase A: freeze backbone params; BN -> eval (re-apply after every .train())."""
    n = 0
    for name, mod in m.named_modules():
        if name.startswith('backbone') and isinstance(mod, torch.nn.BatchNorm2d):
            mod.eval(); n += 1
    for name, p_ in m.named_parameters():
        if name.startswith('backbone'):
            p_.requires_grad_(False)
    return n

if FREEZE_BACKBONE:
    assert BACKBONE_INIT == 'riceseg', 'freezing only makes sense on an in-domain pretrained backbone'
    n_bn = freeze_backbone_(model)
    print(f'T7: backbone FROZEN ({n_bn} BN modules -> eval; params requires_grad=False)')

print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M total | '
      f'{sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M trainable')

optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=BASE_LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scaler    = torch.amp.GradScaler('cuda', enabled=USE_AMP)
ema_model = copy.deepcopy(model).eval() if USE_EMA else None
if ema_model is not None:
    for p in ema_model.parameters():
        p.requires_grad_(False)

@torch.no_grad()
def update_ema_(ema, m, decay):
    msd = m.state_dict()
    for k, v in ema.state_dict().items():
        src = msd[k].detach()
        v.copy_(v * decay + src * (1.0 - decay)) if v.dtype.is_floating_point else v.copy_(src)

total_steps  = max(1, NUM_EPOCHS * len(train_loader))
WARMUP_ITERS = max(1, int(WARMUP_RATIO * total_steps))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(1, total_steps - WARMUP_ITERS), eta_min=MIN_LR)
print(f'{total_steps} steps | warmup {WARMUP_ITERS}')

In [ ]:
# Cell 7 — Train (checkpoints carry RUN_TAG; abort criteria in TEST_LOG.md T1)
train_losses, ap_history = [], []
best_ap50, global_step = -1.0, 0
print(f'Training {NUM_EPOCHS} ep | {len(train_loader)} batches/ep | init={BACKBONE_INIT} | tag={RUN_TAG}\n')

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    if BACKBONE_INIT == 'imagenet':
        wd.apply_bn_policy(model, pretrained_loaded=True)   # after EVERY .train()
    if FREEZE_BACKBONE:
        freeze_backbone_(model)                             # BN back to eval after EVERY .train()

    epoch_loss, n_batches = 0.0, 0
    for imgs, targets in train_loader:
        if not imgs: continue
        imgs    = torch.stack([i.to(device) for i in imgs])
        targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v
                    for k, v in t.items()} for t in targets]
        if global_step < WARMUP_ITERS:
            warmup_lr = BASE_LR * (WARMUP_FACTOR + (1 - WARMUP_FACTOR) * global_step / WARMUP_ITERS)
            for pg in optimizer.param_groups: pg['lr'] = warmup_lr
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            loss_dict = model(imgs, targets)
            loss = loss_dict.get('total_loss', sum(loss_dict.values()))
        assert torch.isfinite(loss), f'non-finite loss at step {global_step} — see TEST_LOG T1 risk #3 (AMP)'
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], max_norm=GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        if ema_model is not None:
            update_ema_(ema_model, model, EMA_DECAY)
        if global_step >= WARMUP_ITERS:
            scheduler.step()
        epoch_loss += loss.item(); n_batches += 1; global_step += 1

    train_losses.append(epoch_loss / max(n_batches, 1))
    line = f'Epoch {epoch:02d}/{NUM_EPOCHS}  train={train_losses[-1]:.4f}  lr={optimizer.param_groups[0]["lr"]:.6f}'

    if epoch % AP_EVAL_EVERY == 0 or epoch == NUM_EPOCHS:
        ap50, ap75, ap5095, dpi = val_eval(ema_model if ema_model is not None else model)
        ap_history.append((epoch, ap50, ap75, dpi))
        line += f'  |  AP@50={ap50:.4f}  AP@75={ap75:.4f}  det/img={dpi:.1f}'
        if ap50 > best_ap50:
            best_ap50 = ap50
            torch.save({'epoch': epoch, 'ap50': ap50, 'ap75': ap75,
                        'state_dict': (ema_model or model).state_dict(),
                        'raw_state_dict': model.state_dict(),
                        'config': {'version': 'v7.1', 'run_tag': RUN_TAG,
                                   'backbone_init': BACKBONE_INIT,
                                   'atss_all_neg': True, 'cls_hard_target': True,
                                   'anchor_base_scale': ANCHOR_BASE_SCALE,
                                   'use_atss': USE_ATSS, 'base_lr': BASE_LR,
                                   'class_names': list(CLASS_NAMES), 'seed': SEED,
                                   'dataset': 'rice_detection_coco_split 80/10/10 dHash-safe'}},
                       BEST_CKPT)
            line += '  * best saved'
    print(line)

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch': epoch, 'state_dict': (ema_model or model).state_dict()},
                   f'{CKPT_DIR}/{RUN_NAME}_epoch{epoch}.pth')

print(f'\nDone. Best val AP@50: {best_ap50:.4f}\n{BEST_CKPT}')
for ep, a50, a75, dpi in ap_history:
    print(f'  epoch {ep:02d}: AP@50={a50:.4f}  AP@75={a75:.4f}  det/img={dpi:.1f}')
print('\n>> Log Actual + Verdict in TEST_LOG.md before touching anything else.')

In [ ]:
# Cell 8 — Curves (guarded)
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(range(1, len(train_losses) + 1), train_losses, marker='o', color='steelblue')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Train loss')
ax1.set_title(f'{RUN_NAME} — Train Loss'); ax1.grid(True, alpha=0.3)

if ap_history:
    eps  = [e for e, *_ in ap_history]
    a50s = [a for _, a, _, _ in ap_history]
    a75s = [a for _, _, a, _ in ap_history]
    ax2.plot(eps, a50s, marker='o', color='seagreen', label='val AP@50')
    ax2.plot(eps, a75s, marker='s', color='darkorange', label='val AP@75')
    bi = a50s.index(max(a50s))
    ax2.axvline(eps[bi], color='seagreen', linestyle=':', alpha=0.6,
                label=f'best (ep {eps[bi]}, {a50s[bi]:.3f})')
    ax2.legend()
else:
    ax2.text(0.5, 0.5, 'no AP evals yet', ha='center')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('AP'); ax2.set_title(f'{RUN_NAME} — Val AP')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/curves_{RUN_NAME}.png', dpi=150)
plt.show()

In [ ]:
# Cell 9 — Final eval on the best checkpoint (valid until paper time; 'test' EXACTLY ONCE)
EVAL_SPLIT = 'valid'

ckpt = torch.load(BEST_CKPT, map_location=device, weights_only=False)
eval_model = wd.WeedDet(num_classes=NUM_CLASSES, anchor_base_scale=ANCHOR_BASE_SCALE,
                        lsc_k=LSC_K, use_atss=USE_ATSS).to(device)
eval_model.load_state_dict(ckpt['state_dict'], strict=True)
print(f"Loaded {BEST_CKPT}  (epoch {ckpt['epoch']}, val AP@50={ckpt.get('ap50', float('nan')):.4f})")

eval_ds = val_ds if EVAL_SPLIT == 'valid' else wd.CocoWeedDataset(
    DATA_ROOT, EVAL_SPLIT, img_size=IMG_SIZE, augment=False, class_names=CLASS_NAMES)
evaluator = val_eval if EVAL_SPLIT == 'valid' else make_evaluator(eval_ds)

ap50, ap75, ap5095, dpi = evaluator(eval_model, verbose=True)
print(f'[{BACKBONE_INIT} | {RUN_TAG} | {EVAL_SPLIT}]  AP@0.50={ap50:.4f}  '
      f'AP@0.75={ap75:.4f}  AP@[.5:.95]={ap5095:.4f}  det/img={dpi:.1f}')
print('Log this row in RESEARCH_PLAN_DETECTION_ACCURACY.md Part C + TEST_LOG.md.')

In [ ]:
# Cell 10 — Raw vs EMA comparison (diagnostic; both live in the best checkpoint)
raw_model = wd.WeedDet(num_classes=NUM_CLASSES, anchor_base_scale=ANCHOR_BASE_SCALE,
                       lsc_k=LSC_K, use_atss=USE_ATSS).to(device)
raw_model.load_state_dict(ckpt['raw_state_dict'], strict=True)

r50, r75, _, rdpi = val_eval(raw_model)
e50, e75, _, edpi = val_eval(eval_model)
print(f'raw : AP@50={r50:.4f}  AP@75={r75:.4f}  det/img={rdpi:.1f}')
print(f'EMA : AP@50={e50:.4f}  AP@75={e75:.4f}  det/img={edpi:.1f}')

In [ ]:
# Cell 11 — Visualize predictions (GT green / pred red)
plot_predictions(eval_model, val_ds, num_samples=3, title=f'EMA {RUN_NAME}')
plot_predictions(raw_model,  val_ds, num_samples=3, title=f'raw {RUN_NAME}')